# Cross-trait and multi-polytranscriptional risk score analysis of PD in AMP-PD: XGBoost multi-PTS models (no PGS)

**Project**: Cross-trait and multi-polytranscriptomic score analysis of Parkinson's disease identifies novel associations and improves prediction

**Date last updated**: July 2026 

 # Initial set-up 

## Loading Python libraries

## Install MLstatkit for Delongs

In [ ]:
!pip install MLstatkit

## Install SHAP

In [ ]:
!pip install shap

In [ ]:
# Use the os package to interact with the environment
import os
import sys

# Bring in Pandas for Dataframe functionality
import pandas as pd
from functools import reduce

# Bring some visualization functionality 
import seaborn as sns  

# numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline

# Enable interaction with the FireCloud API
from firecloud import api as fapi

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.core.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

#Import Sys
import sys as sys

from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, make_scorer
from xgboost import XGBClassifier
from scipy.stats import uniform, randint
from sklearn.base import BaseEstimator, ClassifierMixin
from scipy.stats import mode
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy import stats
from MLstatkit import Delong_test
import shap
from sklearn.metrics import accuracy_score

import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
#pandas2ri.activate()
#from rpy2.robjects.packages import importr
#pROC = importr('pROC')

## Defining functions

In [ ]:
# Utility routine for printing a shell command before executing it
def shell_do(command):
    print(f'Executing: {command}', file=sys.stderr)
    !$command
    
def shell_return(command):
    print(f'Executing: {command}', file=sys.stderr)
    output = !$command
    return '\n'.join(output)

# Utility routine for printing a query before executing it
def bq_query(query):
    print(f'Executing: {query}', file=sys.stderr)
    return pd.read_gbq(query, project_id=BILLING_PROJECT_ID, dialect='standard')

# Utility routine for display a message and a link
def display_html_link(description, link_text, url):
    html = f'''
    <p>
    </p>
    <p>
    {description}
    <a target=_blank href="{url}">{link_text}</a>.
    </p>
    '''

    display(HTML(html))

# Utility routines for reading files from Google Cloud Storage
def gcs_read_file(path):
    """Return the contents of a file in GCS"""
    contents = !gsutil -u {BILLING_PROJECT_ID} cat {path}
    return '\n'.join(contents)
    
def gcs_read_csv(path, sep=None):
    """Return a DataFrame from the contents of a delimited file in GCS"""
    return pd.read_csv(StringIO(gcs_read_file(path)), sep=sep, engine='python')

# Utility routine for displaying a message and link to Cloud Console
def link_to_cloud_console_gcs(description, link_text, gcs_path):
    url = '{}?{}'.format(
        os.path.join('https://console.cloud.google.com/storage/browser',
                     gcs_path.replace("gs://","")),
        urllib.parse.urlencode({'userProject': BILLING_PROJECT_ID}))

    display_html_link(description, link_text, url)

## Set paths

In [ ]:
# Set up billing project and data path variables
BILLING_PROJECT_ID = os.environ['GOOGLE_PROJECT']
WORKSPACE_NAMESPACE = os.environ['WORKSPACE_NAMESPACE']
WORKSPACE_NAME = os.environ['WORKSPACE_NAME']
WORKSPACE_BUCKET = os.environ['WORKSPACE_BUCKET']
WORKSPACE_ATTRIBUTES = fapi.get_workspace(WORKSPACE_NAMESPACE, WORKSPACE_NAME).json().get('workspace',{}).get('attributes',{})

## Print the information to check we are in the proper release and billing 
## This will be different for you, the user, depending on the billing project your workspace is on
print('Billing and Workspace')
print(f'Workspace Name @ `WORKSPACE_NAME`: {WORKSPACE_NAME}')
print(f'Billing Project @ `BILLING_PROJECT_ID`: {BILLING_PROJECT_ID}')
print(f'Workspace Bucket, where you can upload and download data @ `WORKSPACE_BUCKET`: {WORKSPACE_BUCKET}')
print('')

## AMP-PD v4.0
# Explicitly define release v4.0 path 
AMP_RELEASE_CASE_CONTROL_PATH = 'gs://path/removed'
AMP_PPMI_PDBP_TRANSCRIPTOMICS_RELEASE_PATH = 'gs://path/removed'
AMP_HBS_TRANSCRIPTOMICS_RELEASE_PATH = 'gs://path/removed'

#rnaseq_WB-RWTS-VHBS_samples.csv
#rnaseq_WB-RWTS_samples.csv


print('AMP-PD v4.0')
print(f'Path to AMP-PD v4.0 case/control data: {AMP_RELEASE_CASE_CONTROL_PATH}')
print(f'Path to AMP-PD v4.0 PPMI and PDBP RNA Data: {AMP_PPMI_PDBP_TRANSCRIPTOMICS_RELEASE_PATH}')
print(f'Path to AMP-PD v4.0 HBS Data: {AMP_HBS_TRANSCRIPTOMICS_RELEASE_PATH}')

## Check directories

In [ ]:
%%bash
ls /home/jupyter/multiTRS/

# Make a directories for the scorefiles
ls /home/jupyter/multiTRS/multi_score_output/

# Multi-TRS XGBoost models

Here we want to evaluate the general multi-TRS strategy within PDBP and extract the best hyperparameters for fitting a final model and testing externally. 

We can then test this final model in PPMI and HBS

In [ ]:
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, make_scorer
from xgboost import XGBClassifier
from scipy.stats import uniform, randint, mode
from sklearn.base import BaseEstimator, ClassifierMixin
from scipy.stats import mode
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy import stats
from MLstatkit import Delong_test
import shap
from sklearn.metrics import accuracy_score

## SMR-multi scores

### Nested-CV

In [ ]:
# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

# =========================
# Load and join data
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[[
    "participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"
]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "SMR" in c and "single_SNP" not in c]
TRS = TRS[TRS_cols]


combined_data = pd.merge(clinical, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Hyperparameter grid
# =========================
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700, 1000],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [5, 10, 20, 30, 40],
    'colsample_bytree': [0.4, 0.6, 0.8, 1],
    'subsample': [0.6, 0.8],
    'learning_rate': [0.001, 0.0015, 0.01, 0.015, 0.1],
    'gamma': [0, 0.1, 0.3, 0.5, 0.8, 1.0]
}

# =========================
# Nested CV
# =========================
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

outer_auc = []
outer_sensitivity = []
outer_specificity = []
outer_accuracy = []

best_params_list = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):
    print(f"\n--- Outer fold {fold} ---")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(eval_metric='auc', random_state=1)

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=100,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1,
        random_state=1
    )

    random_search.fit(X_train, y_train)

    best_params_list.append(random_search.best_params_)

    # Predictions
    y_prob = random_search.best_estimator_.predict_proba(X_test)[:, 1]

    # AUC
    auc = roc_auc_score(y_test, y_prob)
    outer_auc.append(auc)

    # closest.topleft threshold
    thresh, sens, spec = closest_topleft_threshold(y_test, y_prob)

    y_pred = (y_prob >= thresh).astype(int)
    acc = (y_pred == y_test).mean()

    outer_sensitivity.append(sens)
    outer_specificity.append(spec)
    outer_accuracy.append(acc)

    print(f"AUC: {auc:.4f}, Sens: {sens:.3f}, Spec: {spec:.3f}, Acc: {acc:.3f}")

print("\nNested CV results:")
print(f"AUC mean: {np.mean(outer_auc):.4f} ± {np.std(outer_auc):.4f}")

# =========================
# Aggregate hyperparameters
# =========================
best_params_df = pd.DataFrame(best_params_list)
best_params_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_nestedcv_hyperparameters_all_outer.txt",
                       sep="\t", index=False)
print("\nHyperparameters saved to: /home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_nestedcv_hyperparameters_all_outer.txt")
print(best_params_df)

final_params = {
    'n_estimators': int(mode(best_params_df['n_estimators'], keepdims=True).mode[0]),
    'max_depth': int(mode(best_params_df['max_depth'], keepdims=True).mode[0]),
    'min_child_weight': int(mode(best_params_df['min_child_weight'], keepdims=True).mode[0]),

    'subsample': best_params_df['subsample'].median(),
    'colsample_bytree': best_params_df['colsample_bytree'].median(),
    'learning_rate': best_params_df['learning_rate'].median(),
    'gamma': best_params_df['gamma'].median()
}

# convert dict → single-row DataFrame (FIXED)
final_params_df = pd.DataFrame([final_params])

# save DataFrame (FIXED: was incorrectly using dict)
final_params_df.to_csv(
    "/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_nestedcv_hyperparameters_best.txt",
    sep="\t",
    index=False
)

print("\nHyperparameters saved to:")
print("/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_nestedcv_hyperparameters_best.txt")

print("\nFinal aggregated hyperparameters:")
print(final_params_df)
# Snap continuous params to grid
#for param in ['subsample', 'colsample_bytree', 'learning_rate', 'gamma']:
 #   final_params[param] = min(param_dist[param], key=lambda x: abs(x - final_params[param]))


print("\nFinal aggregated hyperparameters:")
print(final_params)

# =========================
# Null model (baseline)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline"]].copy()

scaler = StandardScaler()
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[["age_at_baseline"]] = scaler.fit_transform(
    X_baseline[["age_at_baseline"]]
)

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky'
)

baseline_model.fit(X_baseline_scaled, y)

y_prob_null = baseline_model.predict_proba(X_baseline_scaled)[:, 1]

auc_null = roc_auc_score(y, y_prob_null)

# closest.topleft for null
null_thresh, null_sens, null_spec = closest_topleft_threshold(y, y_prob_null)

y_pred_null = (y_prob_null >= null_thresh).astype(int)
null_acc = (y_pred_null == y).mean()

print("\nNull model performance:")
print(f"AUC: {auc_null:.4f}, Sens: {null_sens:.3f}, Spec: {null_spec:.3f}, Acc: {null_acc:.3f}")

# =========================
# Final performance table
# =========================
performance_results = pd.DataFrame([{
    "cohort": "PDBP",
    "scores": "SMR-multi",
    "model": "XGBoost",

    "aggregated_params": str(final_params),

    "AUC_null": auc_null,
    "AUC_full_mean_cv": np.mean(outer_auc),
    "AUC_full_sd_cv": np.std(outer_auc),
    "AUC_diff": np.mean(outer_auc) - auc_null,

    "sens_null": null_sens,
    "sens_full_mean_cv": np.mean(outer_sensitivity),
    "sens_full_sd_cv": np.std(outer_sensitivity),
    "sens_diff": np.mean(outer_sensitivity) - null_sens,

    "spec_null": null_spec,
    "spec_full_mean_cv": np.mean(outer_specificity),
    "spec_full_sd_cv": np.std(outer_specificity),
    "spec_diff": np.mean(outer_specificity) - null_spec,

    "acc_null": null_acc,
    "acc_full_mean_cv": np.mean(outer_accuracy),
    "acc_full_sd_cv": np.std(outer_accuracy),
    "acc_diff": np.mean(outer_accuracy) - null_acc
}])

print("\n Performance summary:")
print(performance_results)

# Optional save
performance_results.to_csv("/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_nestedcv.txt", sep="\t", index=False)

### Test in PPMI and HBS

In [ ]:
# =========================
# Load and join data from PDBP
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[["participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "SMR" in c and "single_SNP" not in c]
TRS = TRS[TRS_cols]

combined_data = pd.merge(clinical, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Load saved hyperparameters and aggregate
# =========================
best_params_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_nestedcv_hyperparameters_all_outer.txt", sep="\t")

performance_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_nestedcv.txt", sep="\t")
print(f"\nLoaded mean nested CV AUC: {performance_df['AUC_full_mean_cv'].values[0]:.4f} SD: {performance_df['AUC_full_sd_cv'].values[0]:.4f}")

final_params = {
    'n_estimators':    int(mode(best_params_df['n_estimators'],    keepdims=True).mode[0]),
    'max_depth':       int(mode(best_params_df['max_depth'],       keepdims=True).mode[0]),
    'min_child_weight':int(mode(best_params_df['min_child_weight'],keepdims=True).mode[0]),
    'subsample':       best_params_df['subsample'].median(),
    'colsample_bytree':best_params_df['colsample_bytree'].median(),
    'learning_rate':   best_params_df['learning_rate'].median(),
    'gamma':           best_params_df['gamma'].median()
}

print("\nAggregated final hyperparameters:", final_params)

# =========================
# Fit final XGBoost model on full PDBP
# =========================
final_model = XGBClassifier(
    eval_metric='auc',
    random_state=1,
    **final_params
)

final_model.fit(X, y)

print("\nFinal XGBoost model fitted on full PDBP dataset.")

y_pred_prob = final_model.predict_proba(X)[:, 1]
final_auc = roc_auc_score(y, y_pred_prob)
print(f"Final XGBoost model AUC on full PDBP dataset: {final_auc:.4f}")

# =========================
# Extract and save SHAP values
# =========================
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)
base_value = explainer.expected_value

print("\nSHAP values extracted from final XGBoost model.")
print(f"SHAP values shape: {shap_values.shape}")
print(f"Base value (expected model output): {base_value:.6f}")

shap_df = pd.DataFrame(shap_values, columns=X.columns)
shap_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_PDBP_SHAP_values.txt",
               sep="\t", index=False)
print("SHAP values saved to: /home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_PDBP_SHAP_values.txt")

print("\nGenerating SHAP bar plot...")
plt.figure()
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_PDBP_SHAP_bar_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("SHAP bar plot saved to: /home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_PDBP_SHAP_bar_plot.png")

print("\nGenerating SHAP beeswarm plot...")
plt.figure()
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_PDBP_SHAP_beeswarm_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Beeswarm plot saved to: /home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_PDBP_SHAP_beeswarm_plot.png")

# =========================
# Fit baseline logistic regression model on PDBP (sex + age)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline"]].copy()

scaler = StandardScaler()
cols_to_scale_baseline = ["age_at_baseline"]
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[cols_to_scale_baseline] = scaler.fit_transform(X_baseline[cols_to_scale_baseline])

age_mean = scaler.mean_[0]
age_sd   = scaler.scale_[0]

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky',
    fit_intercept=True
)
baseline_model.fit(X_baseline_scaled, y)

print("\nBaseline logistic regression model fitted on full PDBP dataset.")
print(f"Age scaling parameters - Mean: {age_mean:.4f}, SD: {age_sd:.4f}")
print(f"Baseline model intercept: {baseline_model.intercept_[0]:.6f}")
print(f"Baseline model coefficients (sex, age): {baseline_model.coef_[0]}")

y_baseline_pred_prob = baseline_model.predict_proba(X_baseline_scaled)[:, 1]
baseline_auc = roc_auc_score(y, y_baseline_pred_prob)
print(f"Baseline (sex + age) model AUC on full PDBP dataset: {baseline_auc:.4f}")

# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

def get_accuracy_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return accuracy_score(y_true, y_pred)

# =========================
# External validation in PPMI and HBS
# =========================
external_datasets = {
    "PPMI": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt"
    },
    "HBS": {
        "TRS_path": "/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt"
    }
}

print("\n" + "="*70)
print("EXTERNAL VALIDATION RESULTS WITH DELONG'S TEST")
print("="*70)

full_results_summary = []

for dataset_name, paths in external_datasets.items():
    print(f"\n--- {dataset_name} Dataset ---")

    # Load TRS
    ext_TRS = pd.read_csv(paths["TRS_path"], sep="\t")
    TRS_cols = ["participant_id"] + [c for c in ext_TRS.columns if "SMR" in c and "single_SNP" not in c]
    ext_TRS = ext_TRS[TRS_cols]

    # Merge
    ext_combined = pd.merge(clinical, ext_TRS, on="participant_id", how="inner")
    print(f"Rows in combined data for {dataset_name}: {len(ext_combined)}")

    # Predictions - full model
    X_ext = ext_combined[X.columns]
    y_ext = ext_combined["case_control_other_at_baseline"]
    y_ext_pred_prob = final_model.predict_proba(X_ext)[:, 1]

    # Predictions - baseline model (scaled with PDBP parameters)
    X_ext_baseline = ext_combined[["sex", "age_at_baseline"]].copy()
    X_ext_baseline_scaled = X_ext_baseline.copy()
    X_ext_baseline_scaled["age_at_baseline"] = (X_ext_baseline["age_at_baseline"] - age_mean) / age_sd
    y_ext_baseline_pred_prob = baseline_model.predict_proba(X_ext_baseline_scaled)[:, 1]

    # DeLong test
    z_stat, p_value, ci_full, ci_null, auc_full, auc_null, info = Delong_test(
        y_ext, y_ext_pred_prob, y_ext_baseline_pred_prob,
        return_ci=True, return_auc=True, verbose=0
    )

    print(f"XGBoost model AUC:                   {auc_full:.4f} (95% CI: {ci_full[0]:.4f}-{ci_full[1]:.4f})")
    print(f"Baseline model AUC:                  {auc_null:.4f} (95% CI: {ci_null[0]:.4f}-{ci_null[1]:.4f})")
    print(f"AUC Difference (XGBoost - Baseline): {auc_full - auc_null:.4f}")
    print(f"Variance of AUC difference:          {info['var_diff']:.6f}")
    print(f"DeLong's test Z-statistic:           {z_stat:.4f}")
    print(f"DeLong's test p-value (2-tailed):    {p_value:.4e}")
    print(f"Result: {'Significantly different (p < 0.05)' if p_value < 0.05 else 'Not significantly different (p >= 0.05)'}")

    # Optimal threshold metrics
    thresh_full, sens_full, spec_full = closest_topleft_threshold(y_ext, y_ext_pred_prob)
    thresh_null, sens_null, spec_null = closest_topleft_threshold(y_ext, y_ext_baseline_pred_prob)

    acc_full = get_accuracy_at_threshold(y_ext, y_ext_pred_prob,          thresh_full)
    acc_null = get_accuracy_at_threshold(y_ext, y_ext_baseline_pred_prob, thresh_null)

    full_results_summary.append({
        'cohort':         dataset_name,
        'AUC_null':       auc_null,
        'AUC_lower_null': ci_null[0],
        'AUC_upper_null': ci_null[1],
        'AUC_full':       auc_full,
        'AUC_lower_full': ci_full[0],
        'AUC_upper_full': ci_full[1],
        'AUC_diff':       auc_full - auc_null,
        'sens_null':      sens_null,
        'sens_full':      sens_full,
        'sens_diff':      sens_full - sens_null,
        'spec_null':      spec_null,
        'spec_full':      spec_full,
        'spec_diff':      spec_full - spec_null,
        'acc_null':       acc_null,
        'acc_full':       acc_full,
        'acc_diff':       acc_full - acc_null,
        'delong_z':       z_stat,
        'delong_p':       p_value
    })

# =========================
# Save full results table
# =========================
print("\n" + "="*70)
print("FULL RESULTS TABLE")
print("="*70)
full_results_df = pd.DataFrame(full_results_summary)
print(full_results_df.to_string(index=False))

full_results_outfile = "/home/jupyter/multiTRS/results/multi_TRS_SMR_multi_XGBoost_external_validation_full_results.txt"
full_results_df.to_csv(full_results_outfile, sep="\t", index=False)
print(f"\nFull results saved to: {full_results_outfile}")

## SMR single SNP

### Nested-CV

In [ ]:
# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

# =========================
# Load and join data
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[[
    "participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"
]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "single_SNP" in c and "Hip_Fracture" not in c]
TRS = TRS[TRS_cols]


combined_data = pd.merge(clinical, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Hyperparameter grid
# =========================
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700, 1000],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [5, 10, 20, 30, 40],
    'colsample_bytree': [0.4, 0.6, 0.8, 1],
    'subsample': [0.6, 0.8],
    'learning_rate': [0.001, 0.0015, 0.01, 0.015, 0.1],
    'gamma': [0, 0.1, 0.3, 0.5, 0.8, 1.0]
}

# =========================
# Nested CV
# =========================
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

outer_auc = []
outer_sensitivity = []
outer_specificity = []
outer_accuracy = []

best_params_list = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):
    print(f"\n--- Outer fold {fold} ---")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(eval_metric='auc', random_state=1)

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=100,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1,
        random_state=1
    )

    random_search.fit(X_train, y_train)

    best_params_list.append(random_search.best_params_)

    # Predictions
    y_prob = random_search.best_estimator_.predict_proba(X_test)[:, 1]

    # AUC
    auc = roc_auc_score(y_test, y_prob)
    outer_auc.append(auc)

    # closest.topleft threshold
    thresh, sens, spec = closest_topleft_threshold(y_test, y_prob)

    y_pred = (y_prob >= thresh).astype(int)
    acc = (y_pred == y_test).mean()

    outer_sensitivity.append(sens)
    outer_specificity.append(spec)
    outer_accuracy.append(acc)

    print(f"AUC: {auc:.4f}, Sens: {sens:.3f}, Spec: {spec:.3f}, Acc: {acc:.3f}")

print("\nNested CV results:")
print(f"AUC mean: {np.mean(outer_auc):.4f} ± {np.std(outer_auc):.4f}")

# =========================
# Aggregate hyperparameters
# =========================
best_params_df = pd.DataFrame(best_params_list)
best_params_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_nestedcv_hyperparameters_all_outer.txt",
                       sep="\t", index=False)
print("\nHyperparameters saved to: /home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_nestedcv_hyperparameters_all_outer.txt")
print(best_params_df)

final_params = {
    'n_estimators': int(mode(best_params_df['n_estimators'], keepdims=True).mode[0]),
    'max_depth': int(mode(best_params_df['max_depth'], keepdims=True).mode[0]),
    'min_child_weight': int(mode(best_params_df['min_child_weight'], keepdims=True).mode[0]),

    'subsample': best_params_df['subsample'].median(),
    'colsample_bytree': best_params_df['colsample_bytree'].median(),
    'learning_rate': best_params_df['learning_rate'].median(),
    'gamma': best_params_df['gamma'].median()
}

# convert dict → single-row DataFrame (FIXED)
final_params_df = pd.DataFrame([final_params])

# save DataFrame (FIXED: was incorrectly using dict)
final_params_df.to_csv(
    "/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_nestedcv_hyperparameters_best.txt",
    sep="\t",
    index=False
)

print("\nHyperparameters saved to:")
print("/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_nestedcv_hyperparameters_best.txt")

print("\nFinal aggregated hyperparameters:")
print(final_params_df)
# Snap continuous params to grid
#for param in ['subsample', 'colsample_bytree', 'learning_rate', 'gamma']:
 #   final_params[param] = min(param_dist[param], key=lambda x: abs(x - final_params[param]))


print("\nFinal aggregated hyperparameters:")
print(final_params)

# =========================
# Null model (baseline)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline"]].copy()

scaler = StandardScaler()
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[["age_at_baseline"]] = scaler.fit_transform(
    X_baseline[["age_at_baseline"]]
)

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky'
)

baseline_model.fit(X_baseline_scaled, y)

y_prob_null = baseline_model.predict_proba(X_baseline_scaled)[:, 1]

auc_null = roc_auc_score(y, y_prob_null)

# closest.topleft for null
null_thresh, null_sens, null_spec = closest_topleft_threshold(y, y_prob_null)

y_pred_null = (y_prob_null >= null_thresh).astype(int)
null_acc = (y_pred_null == y).mean()

print("\nNull model performance:")
print(f"AUC: {auc_null:.4f}, Sens: {null_sens:.3f}, Spec: {null_spec:.3f}, Acc: {null_acc:.3f}")

# =========================
# Final performance table
# =========================
performance_results = pd.DataFrame([{
    "cohort": "PDBP",
    "scores": "SMR-single-SNP",
    "model": "XGBoost",

    "aggregated_params": str(final_params),

    "AUC_null": auc_null,
    "AUC_full_mean_cv": np.mean(outer_auc),
    "AUC_full_sd_cv": np.std(outer_auc),
    "AUC_diff": np.mean(outer_auc) - auc_null,

    "sens_null": null_sens,
    "sens_full_mean_cv": np.mean(outer_sensitivity),
    "sens_full_sd_cv": np.std(outer_sensitivity),
    "sens_diff": np.mean(outer_sensitivity) - null_sens,

    "spec_null": null_spec,
    "spec_full_mean_cv": np.mean(outer_specificity),
    "spec_full_sd_cv": np.std(outer_specificity),
    "spec_diff": np.mean(outer_specificity) - null_spec,

    "acc_null": null_acc,
    "acc_full_mean_cv": np.mean(outer_accuracy),
    "acc_full_sd_cv": np.std(outer_accuracy),
    "acc_diff": np.mean(outer_accuracy) - null_acc
}])

print("\n Performance summary:")
print(performance_results)

# Optional save
performance_results.to_csv("/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_nestedcv.txt", sep="\t", index=False)

### Test in PPMI and HBS

In [ ]:
# =========================
# Load and join data from PDBP
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[["participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "single_SNP" in c and "Hip_Fracture" not in c]
TRS = TRS[TRS_cols]

combined_data = pd.merge(clinical, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Load saved hyperparameters and aggregate
# =========================
best_params_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_nestedcv_hyperparameters_all_outer.txt", sep="\t")

performance_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_nestedcv.txt", sep="\t")
print(f"\nLoaded mean nested CV AUC: {performance_df['AUC_full_mean_cv'].values[0]:.4f} SD: {performance_df['AUC_full_sd_cv'].values[0]:.4f}")

final_params = {
    'n_estimators':    int(mode(best_params_df['n_estimators'],    keepdims=True).mode[0]),
    'max_depth':       int(mode(best_params_df['max_depth'],       keepdims=True).mode[0]),
    'min_child_weight':int(mode(best_params_df['min_child_weight'],keepdims=True).mode[0]),
    'subsample':       best_params_df['subsample'].median(),
    'colsample_bytree':best_params_df['colsample_bytree'].median(),
    'learning_rate':   best_params_df['learning_rate'].median(),
    'gamma':           best_params_df['gamma'].median()
}

print("\nAggregated final hyperparameters:", final_params)

# =========================
# Fit final XGBoost model on full PDBP
# =========================
final_model = XGBClassifier(
    eval_metric='auc',
    random_state=1,
    **final_params
)

final_model.fit(X, y)

print("\nFinal XGBoost model fitted on full PDBP dataset.")

y_pred_prob = final_model.predict_proba(X)[:, 1]
final_auc = roc_auc_score(y, y_pred_prob)
print(f"Final XGBoost model AUC on full PDBP dataset: {final_auc:.4f}")

# =========================
# Extract and save SHAP values
# =========================
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)
base_value = explainer.expected_value

print("\nSHAP values extracted from final XGBoost model.")
print(f"SHAP values shape: {shap_values.shape}")
print(f"Base value (expected model output): {base_value:.6f}")

shap_df = pd.DataFrame(shap_values, columns=X.columns)
shap_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_PDBP_SHAP_values.txt",
               sep="\t", index=False)
print("SHAP values saved to: /home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_PDBP_SHAP_values.txt")

print("\nGenerating SHAP bar plot...")
plt.figure()
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_PDBP_SHAP_bar_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("SHAP bar plot saved to: /home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_PDBP_SHAP_bar_plot.png")

print("\nGenerating SHAP beeswarm plot...")
plt.figure()
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_PDBP_SHAP_beeswarm_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Beeswarm plot saved to: /home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_PDBP_SHAP_beeswarm_plot.png")

# =========================
# Fit baseline logistic regression model on PDBP (sex + age)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline"]].copy()

scaler = StandardScaler()
cols_to_scale_baseline = ["age_at_baseline"]
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[cols_to_scale_baseline] = scaler.fit_transform(X_baseline[cols_to_scale_baseline])

age_mean = scaler.mean_[0]
age_sd   = scaler.scale_[0]

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky',
    fit_intercept=True
)
baseline_model.fit(X_baseline_scaled, y)

print("\nBaseline logistic regression model fitted on full PDBP dataset.")
print(f"Age scaling parameters - Mean: {age_mean:.4f}, SD: {age_sd:.4f}")
print(f"Baseline model intercept: {baseline_model.intercept_[0]:.6f}")
print(f"Baseline model coefficients (sex, age): {baseline_model.coef_[0]}")

y_baseline_pred_prob = baseline_model.predict_proba(X_baseline_scaled)[:, 1]
baseline_auc = roc_auc_score(y, y_baseline_pred_prob)
print(f"Baseline (sex + age) model AUC on full PDBP dataset: {baseline_auc:.4f}")

# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

def get_accuracy_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return accuracy_score(y_true, y_pred)

# =========================
# External validation in PPMI and HBS
# =========================
external_datasets = {
    "PPMI": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt"
    },
    "HBS": {
        "TRS_path": "/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt"
    }
}

print("\n" + "="*70)
print("EXTERNAL VALIDATION RESULTS WITH DELONG'S TEST")
print("="*70)

full_results_summary = []

for dataset_name, paths in external_datasets.items():
    print(f"\n--- {dataset_name} Dataset ---")

    # Load TRS
    ext_TRS = pd.read_csv(paths["TRS_path"], sep="\t")
    TRS_cols = ["participant_id"] + [c for c in ext_TRS.columns if "single_SNP" in c and "Hip_Fracture" not in c]
    ext_TRS = ext_TRS[TRS_cols]

    # Merge
    ext_combined = pd.merge(clinical, ext_TRS, on="participant_id", how="inner")
    print(f"Rows in combined data for {dataset_name}: {len(ext_combined)}")

    # Predictions - full model
    X_ext = ext_combined[X.columns]
    y_ext = ext_combined["case_control_other_at_baseline"]
    y_ext_pred_prob = final_model.predict_proba(X_ext)[:, 1]

    # Predictions - baseline model (scaled with PDBP parameters)
    X_ext_baseline = ext_combined[["sex", "age_at_baseline"]].copy()
    X_ext_baseline_scaled = X_ext_baseline.copy()
    X_ext_baseline_scaled["age_at_baseline"] = (X_ext_baseline["age_at_baseline"] - age_mean) / age_sd
    y_ext_baseline_pred_prob = baseline_model.predict_proba(X_ext_baseline_scaled)[:, 1]

    # DeLong test
    z_stat, p_value, ci_full, ci_null, auc_full, auc_null, info = Delong_test(
        y_ext, y_ext_pred_prob, y_ext_baseline_pred_prob,
        return_ci=True, return_auc=True, verbose=0
    )

    print(f"XGBoost model AUC:                   {auc_full:.4f} (95% CI: {ci_full[0]:.4f}-{ci_full[1]:.4f})")
    print(f"Baseline model AUC:                  {auc_null:.4f} (95% CI: {ci_null[0]:.4f}-{ci_null[1]:.4f})")
    print(f"AUC Difference (XGBoost - Baseline): {auc_full - auc_null:.4f}")
    print(f"Variance of AUC difference:          {info['var_diff']:.6f}")
    print(f"DeLong's test Z-statistic:           {z_stat:.4f}")
    print(f"DeLong's test p-value (2-tailed):    {p_value:.4e}")
    print(f"Result: {'Significantly different (p < 0.05)' if p_value < 0.05 else 'Not significantly different (p >= 0.05)'}")

    # Optimal threshold metrics
    thresh_full, sens_full, spec_full = closest_topleft_threshold(y_ext, y_ext_pred_prob)
    thresh_null, sens_null, spec_null = closest_topleft_threshold(y_ext, y_ext_baseline_pred_prob)

    acc_full = get_accuracy_at_threshold(y_ext, y_ext_pred_prob,          thresh_full)
    acc_null = get_accuracy_at_threshold(y_ext, y_ext_baseline_pred_prob, thresh_null)

    full_results_summary.append({
        'cohort':         dataset_name,
        'AUC_null':       auc_null,
        'AUC_lower_null': ci_null[0],
        'AUC_upper_null': ci_null[1],
        'AUC_full':       auc_full,
        'AUC_lower_full': ci_full[0],
        'AUC_upper_full': ci_full[1],
        'AUC_diff':       auc_full - auc_null,
        'sens_null':      sens_null,
        'sens_full':      sens_full,
        'sens_diff':      sens_full - sens_null,
        'spec_null':      spec_null,
        'spec_full':      spec_full,
        'spec_diff':      spec_full - spec_null,
        'acc_null':       acc_null,
        'acc_full':       acc_full,
        'acc_diff':       acc_full - acc_null,
        'delong_z':       z_stat,
        'delong_p':       p_value
    })

# =========================
# Save full results table
# =========================
print("\n" + "="*70)
print("FULL RESULTS TABLE")
print("="*70)
full_results_df = pd.DataFrame(full_results_summary)
print(full_results_df.to_string(index=False))

full_results_outfile = "/home/jupyter/multiTRS/results/multi_TRS_SMR_single_SNP_XGBoost_external_validation_full_results.txt"
full_results_df.to_csv(full_results_outfile, sep="\t", index=False)
print(f"\nFull results saved to: {full_results_outfile}")

## FUSION scores

### Nested-CV

In [ ]:
# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

# =========================
# Load and join data
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[[
    "participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"
]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "fusion" in c]
TRS = TRS[TRS_cols]


combined_data = pd.merge(clinical, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Hyperparameter grid
# =========================
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700, 1000],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [5, 10, 20, 30, 40],
    'colsample_bytree': [0.4, 0.6, 0.8, 1],
    'subsample': [0.6, 0.8],
    'learning_rate': [0.001, 0.0015, 0.01, 0.015, 0.1],
    'gamma': [0, 0.1, 0.3, 0.5, 0.8, 1.0]
}

# =========================
# Nested CV
# =========================
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

outer_auc = []
outer_sensitivity = []
outer_specificity = []
outer_accuracy = []

best_params_list = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):
    print(f"\n--- Outer fold {fold} ---")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(eval_metric='auc', random_state=1)

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=100,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1,
        random_state=1
    )

    random_search.fit(X_train, y_train)

    best_params_list.append(random_search.best_params_)

    # Predictions
    y_prob = random_search.best_estimator_.predict_proba(X_test)[:, 1]

    # AUC
    auc = roc_auc_score(y_test, y_prob)
    outer_auc.append(auc)

    # closest.topleft threshold
    thresh, sens, spec = closest_topleft_threshold(y_test, y_prob)

    y_pred = (y_prob >= thresh).astype(int)
    acc = (y_pred == y_test).mean()

    outer_sensitivity.append(sens)
    outer_specificity.append(spec)
    outer_accuracy.append(acc)

    print(f"AUC: {auc:.4f}, Sens: {sens:.3f}, Spec: {spec:.3f}, Acc: {acc:.3f}")

print("\nNested CV results:")
print(f"AUC mean: {np.mean(outer_auc):.4f} ± {np.std(outer_auc):.4f}")

# =========================
# Aggregate hyperparameters
# =========================
best_params_df = pd.DataFrame(best_params_list)
best_params_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_nestedcv_hyperparameters_all_outer.txt",
                       sep="\t", index=False)
print("\nHyperparameters saved to: /home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_nestedcv_hyperparameters_all_outer.txt")
print(best_params_df)

final_params = {
    'n_estimators': int(mode(best_params_df['n_estimators'], keepdims=True).mode[0]),
    'max_depth': int(mode(best_params_df['max_depth'], keepdims=True).mode[0]),
    'min_child_weight': int(mode(best_params_df['min_child_weight'], keepdims=True).mode[0]),

    'subsample': best_params_df['subsample'].median(),
    'colsample_bytree': best_params_df['colsample_bytree'].median(),
    'learning_rate': best_params_df['learning_rate'].median(),
    'gamma': best_params_df['gamma'].median()
}

# convert dict → single-row DataFrame (FIXED)
final_params_df = pd.DataFrame([final_params])

# save DataFrame (FIXED: was incorrectly using dict)
final_params_df.to_csv(
    "/home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_nestedcv_hyperparameters_best.txt",
    sep="\t",
    index=False
)

print("\nHyperparameters saved to:")
print("/home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_nestedcv_hyperparameters_best.txt")

print("\nFinal aggregated hyperparameters:")
print(final_params_df)
# Snap continuous params to grid
#for param in ['subsample', 'colsample_bytree', 'learning_rate', 'gamma']:
 #   final_params[param] = min(param_dist[param], key=lambda x: abs(x - final_params[param]))


print("\nFinal aggregated hyperparameters:")
print(final_params)

# =========================
# Null model (baseline)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline"]].copy()

scaler = StandardScaler()
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[["age_at_baseline"]] = scaler.fit_transform(
    X_baseline[["age_at_baseline"]]
)

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky'
)

baseline_model.fit(X_baseline_scaled, y)

y_prob_null = baseline_model.predict_proba(X_baseline_scaled)[:, 1]

auc_null = roc_auc_score(y, y_prob_null)

# closest.topleft for null
null_thresh, null_sens, null_spec = closest_topleft_threshold(y, y_prob_null)

y_pred_null = (y_prob_null >= null_thresh).astype(int)
null_acc = (y_pred_null == y).mean()

print("\nNull model performance:")
print(f"AUC: {auc_null:.4f}, Sens: {null_sens:.3f}, Spec: {null_spec:.3f}, Acc: {null_acc:.3f}")

# =========================
# Final performance table
# =========================
performance_results = pd.DataFrame([{
    "cohort": "PDBP",
    "scores": "FUSION",
    "model": "XGBoost",

    "aggregated_params": str(final_params),

    "AUC_null": auc_null,
    "AUC_full_mean_cv": np.mean(outer_auc),
    "AUC_full_sd_cv": np.std(outer_auc),
    "AUC_diff": np.mean(outer_auc) - auc_null,

    "sens_null": null_sens,
    "sens_full_mean_cv": np.mean(outer_sensitivity),
    "sens_full_sd_cv": np.std(outer_sensitivity),
    "sens_diff": np.mean(outer_sensitivity) - null_sens,

    "spec_null": null_spec,
    "spec_full_mean_cv": np.mean(outer_specificity),
    "spec_full_sd_cv": np.std(outer_specificity),
    "spec_diff": np.mean(outer_specificity) - null_spec,

    "acc_null": null_acc,
    "acc_full_mean_cv": np.mean(outer_accuracy),
    "acc_full_sd_cv": np.std(outer_accuracy),
    "acc_diff": np.mean(outer_accuracy) - null_acc
}])

print("\n Performance summary:")
print(performance_results)

# Optional save
performance_results.to_csv("/home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_nestedcv.txt", sep="\t", index=False)

### Test in PPMI and HBS

In [ ]:
# =========================
# Load and join data from PDBP
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[["participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = ["participant_id"] + [c for c in TRS.columns if "fusion" in c]
TRS = TRS[TRS_cols]

combined_data = pd.merge(clinical, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Load saved hyperparameters and aggregate
# =========================
best_params_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_nestedcv_hyperparameters_all_outer.txt", sep="\t")

performance_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_nestedcv.txt", sep="\t")
print(f"\nLoaded mean nested CV AUC: {performance_df['AUC_full_mean_cv'].values[0]:.4f} SD: {performance_df['AUC_full_sd_cv'].values[0]:.4f}")

final_params = {
    'n_estimators':    int(mode(best_params_df['n_estimators'],    keepdims=True).mode[0]),
    'max_depth':       int(mode(best_params_df['max_depth'],       keepdims=True).mode[0]),
    'min_child_weight':int(mode(best_params_df['min_child_weight'],keepdims=True).mode[0]),
    'subsample':       best_params_df['subsample'].median(),
    'colsample_bytree':best_params_df['colsample_bytree'].median(),
    'learning_rate':   best_params_df['learning_rate'].median(),
    'gamma':           best_params_df['gamma'].median()
}

print("\nAggregated final hyperparameters:", final_params)

# =========================
# Fit final XGBoost model on full PDBP
# =========================
final_model = XGBClassifier(
    eval_metric='auc',
    random_state=1,
    **final_params
)

final_model.fit(X, y)

print("\nFinal XGBoost model fitted on full PDBP dataset.")

y_pred_prob = final_model.predict_proba(X)[:, 1]
final_auc = roc_auc_score(y, y_pred_prob)
print(f"Final XGBoost model AUC on full PDBP dataset: {final_auc:.4f}")

# =========================
# Extract and save SHAP values
# =========================
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)
base_value = explainer.expected_value

print("\nSHAP values extracted from final XGBoost model.")
print(f"SHAP values shape: {shap_values.shape}")
print(f"Base value (expected model output): {base_value:.6f}")

shap_df = pd.DataFrame(shap_values, columns=X.columns)
shap_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_PDBP_SHAP_values.txt",
               sep="\t", index=False)
print("SHAP values saved to: /home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_PDBP_SHAP_values.txt")

print("\nGenerating SHAP bar plot...")
plt.figure()
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_FUSION_PDBP_SHAP_bar_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("SHAP bar plot saved to: /home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_PDBP_SHAP_bar_plot.png")

print("\nGenerating SHAP beeswarm plot...")
plt.figure()
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_PDBP_SHAP_beeswarm_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Beeswarm plot saved to: /home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_PDBP_SHAP_beeswarm_plot.png")

# =========================
# Fit baseline logistic regression model on PDBP (sex + age)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline"]].copy()

scaler = StandardScaler()
cols_to_scale_baseline = ["age_at_baseline"]
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[cols_to_scale_baseline] = scaler.fit_transform(X_baseline[cols_to_scale_baseline])

age_mean = scaler.mean_[0]
age_sd   = scaler.scale_[0]

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky',
    fit_intercept=True
)
baseline_model.fit(X_baseline_scaled, y)

print("\nBaseline logistic regression model fitted on full PDBP dataset.")
print(f"Age scaling parameters - Mean: {age_mean:.4f}, SD: {age_sd:.4f}")
print(f"Baseline model intercept: {baseline_model.intercept_[0]:.6f}")
print(f"Baseline model coefficients (sex, age): {baseline_model.coef_[0]}")

y_baseline_pred_prob = baseline_model.predict_proba(X_baseline_scaled)[:, 1]
baseline_auc = roc_auc_score(y, y_baseline_pred_prob)
print(f"Baseline (sex + age) model AUC on full PDBP dataset: {baseline_auc:.4f}")

# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

def get_accuracy_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return accuracy_score(y_true, y_pred)

# =========================
# External validation in PPMI and HBS
# =========================
external_datasets = {
    "PPMI": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt"
    },
    "HBS": {
        "TRS_path": "/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt"
    }
}

print("\n" + "="*70)
print("EXTERNAL VALIDATION RESULTS WITH DELONG'S TEST")
print("="*70)

full_results_summary = []

for dataset_name, paths in external_datasets.items():
    print(f"\n--- {dataset_name} Dataset ---")

    # Load TRS
    ext_TRS = pd.read_csv(paths["TRS_path"], sep="\t")
    TRS_cols = ["participant_id"] + [c for c in ext_TRS.columns if "fusion" in c]
    ext_TRS = ext_TRS[TRS_cols]

    # Merge
    ext_combined = pd.merge(clinical, ext_TRS, on="participant_id", how="inner")
    print(f"Rows in combined data for {dataset_name}: {len(ext_combined)}")

    # Predictions - full model
    X_ext = ext_combined[X.columns]
    y_ext = ext_combined["case_control_other_at_baseline"]
    y_ext_pred_prob = final_model.predict_proba(X_ext)[:, 1]

    # Predictions - baseline model (scaled with PDBP parameters)
    X_ext_baseline = ext_combined[["sex", "age_at_baseline"]].copy()
    X_ext_baseline_scaled = X_ext_baseline.copy()
    X_ext_baseline_scaled["age_at_baseline"] = (X_ext_baseline["age_at_baseline"] - age_mean) / age_sd
    y_ext_baseline_pred_prob = baseline_model.predict_proba(X_ext_baseline_scaled)[:, 1]

    # DeLong test
    z_stat, p_value, ci_full, ci_null, auc_full, auc_null, info = Delong_test(
        y_ext, y_ext_pred_prob, y_ext_baseline_pred_prob,
        return_ci=True, return_auc=True, verbose=0
    )

    print(f"XGBoost model AUC:                   {auc_full:.4f} (95% CI: {ci_full[0]:.4f}-{ci_full[1]:.4f})")
    print(f"Baseline model AUC:                  {auc_null:.4f} (95% CI: {ci_null[0]:.4f}-{ci_null[1]:.4f})")
    print(f"AUC Difference (XGBoost - Baseline): {auc_full - auc_null:.4f}")
    print(f"Variance of AUC difference:          {info['var_diff']:.6f}")
    print(f"DeLong's test Z-statistic:           {z_stat:.4f}")
    print(f"DeLong's test p-value (2-tailed):    {p_value:.4e}")
    print(f"Result: {'Significantly different (p < 0.05)' if p_value < 0.05 else 'Not significantly different (p >= 0.05)'}")

    # Optimal threshold metrics
    thresh_full, sens_full, spec_full = closest_topleft_threshold(y_ext, y_ext_pred_prob)
    thresh_null, sens_null, spec_null = closest_topleft_threshold(y_ext, y_ext_baseline_pred_prob)

    acc_full = get_accuracy_at_threshold(y_ext, y_ext_pred_prob,          thresh_full)
    acc_null = get_accuracy_at_threshold(y_ext, y_ext_baseline_pred_prob, thresh_null)

    full_results_summary.append({
        'cohort':         dataset_name,
        'AUC_null':       auc_null,
        'AUC_lower_null': ci_null[0],
        'AUC_upper_null': ci_null[1],
        'AUC_full':       auc_full,
        'AUC_lower_full': ci_full[0],
        'AUC_upper_full': ci_full[1],
        'AUC_diff':       auc_full - auc_null,
        'sens_null':      sens_null,
        'sens_full':      sens_full,
        'sens_diff':      sens_full - sens_null,
        'spec_null':      spec_null,
        'spec_full':      spec_full,
        'spec_diff':      spec_full - spec_null,
        'acc_null':       acc_null,
        'acc_full':       acc_full,
        'acc_diff':       acc_full - acc_null,
        'delong_z':       z_stat,
        'delong_p':       p_value
    })

# =========================
# Save full results table
# =========================
print("\n" + "="*70)
print("FULL RESULTS TABLE")
print("="*70)
full_results_df = pd.DataFrame(full_results_summary)
print(full_results_df.to_string(index=False))

full_results_outfile = "/home/jupyter/multiTRS/results/multi_TRS_FUSION_XGBoost_external_validation_full_results.txt"
full_results_df.to_csv(full_results_outfile, sep="\t", index=False)
print(f"\nFull results saved to: {full_results_outfile}")

## All scores

### Nested-CV

In [ ]:
# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

# =========================
# Load and join data
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[[
    "participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"
]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = [c for c in TRS.columns if "Hip_fracture_EUR_2022_FDR_SMR_single_SNP" not in c]
TRS = TRS[TRS_cols]


combined_data = pd.merge(clinical, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Hyperparameter grid
# =========================
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700, 1000],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [5, 10, 20, 30, 40],
    'colsample_bytree': [0.4, 0.6, 0.8, 1],
    'subsample': [0.6, 0.8],
    'learning_rate': [0.001, 0.0015, 0.01, 0.015, 0.1],
    'gamma': [0, 0.1, 0.3, 0.5, 0.8, 1.0]
}

# =========================
# Nested CV
# =========================
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

outer_auc = []
outer_sensitivity = []
outer_specificity = []
outer_accuracy = []

best_params_list = []

for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y), 1):
    print(f"\n--- Outer fold {fold} ---")

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(eval_metric='auc', random_state=1)

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=100,
        scoring="roc_auc",
        cv=5,
        n_jobs=-1,
        random_state=1
    )

    random_search.fit(X_train, y_train)

    best_params_list.append(random_search.best_params_)

    # Predictions
    y_prob = random_search.best_estimator_.predict_proba(X_test)[:, 1]

    # AUC
    auc = roc_auc_score(y_test, y_prob)
    outer_auc.append(auc)

    # closest.topleft threshold
    thresh, sens, spec = closest_topleft_threshold(y_test, y_prob)

    y_pred = (y_prob >= thresh).astype(int)
    acc = (y_pred == y_test).mean()

    outer_sensitivity.append(sens)
    outer_specificity.append(spec)
    outer_accuracy.append(acc)

    print(f"AUC: {auc:.4f}, Sens: {sens:.3f}, Spec: {spec:.3f}, Acc: {acc:.3f}")

print("\nNested CV results:")
print(f"AUC mean: {np.mean(outer_auc):.4f} ± {np.std(outer_auc):.4f}")

# =========================
# Aggregate hyperparameters
# =========================
best_params_df = pd.DataFrame(best_params_list)
best_params_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_nestedcv_hyperparameters_all_outer.txt",
                       sep="\t", index=False)
print("\nHyperparameters saved to: /home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_nestedcv_hyperparameters_all_outer.txt")
print(best_params_df)

final_params = {
    'n_estimators': int(mode(best_params_df['n_estimators'], keepdims=True).mode[0]),
    'max_depth': int(mode(best_params_df['max_depth'], keepdims=True).mode[0]),
    'min_child_weight': int(mode(best_params_df['min_child_weight'], keepdims=True).mode[0]),

    'subsample': best_params_df['subsample'].median(),
    'colsample_bytree': best_params_df['colsample_bytree'].median(),
    'learning_rate': best_params_df['learning_rate'].median(),
    'gamma': best_params_df['gamma'].median()
}

# convert dict → single-row DataFrame (FIXED)
final_params_df = pd.DataFrame([final_params])

# save DataFrame (FIXED: was incorrectly using dict)
final_params_df.to_csv(
    "/home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_nestedcv_hyperparameters_best.txt",
    sep="\t",
    index=False
)

print("\nHyperparameters saved to:")
print("/home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_nestedcv_hyperparameters_best.txt")

print("\nFinal aggregated hyperparameters:")
print(final_params_df)
# Snap continuous params to grid
#for param in ['subsample', 'colsample_bytree', 'learning_rate', 'gamma']:
 #   final_params[param] = min(param_dist[param], key=lambda x: abs(x - final_params[param]))


print("\nFinal aggregated hyperparameters:")
print(final_params)

# =========================
# Null model (baseline)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline"]].copy()

scaler = StandardScaler()
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[["age_at_baseline"]] = scaler.fit_transform(
    X_baseline[["age_at_baseline"]]
)

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky'
)

baseline_model.fit(X_baseline_scaled, y)

y_prob_null = baseline_model.predict_proba(X_baseline_scaled)[:, 1]

auc_null = roc_auc_score(y, y_prob_null)

# closest.topleft for null
null_thresh, null_sens, null_spec = closest_topleft_threshold(y, y_prob_null)

y_pred_null = (y_prob_null >= null_thresh).astype(int)
null_acc = (y_pred_null == y).mean()

print("\nNull model performance:")
print(f"AUC: {auc_null:.4f}, Sens: {null_sens:.3f}, Spec: {null_spec:.3f}, Acc: {null_acc:.3f}")

# =========================
# Final performance table
# =========================
performance_results = pd.DataFrame([{
    "cohort": "PDBP",
    "scores": "all_scores",
    "model": "XGBoost",

    "aggregated_params": str(final_params),

    "AUC_null": auc_null,
    "AUC_full_mean_cv": np.mean(outer_auc),
    "AUC_full_sd_cv": np.std(outer_auc),
    "AUC_diff": np.mean(outer_auc) - auc_null,

    "sens_null": null_sens,
    "sens_full_mean_cv": np.mean(outer_sensitivity),
    "sens_full_sd_cv": np.std(outer_sensitivity),
    "sens_diff": np.mean(outer_sensitivity) - null_sens,

    "spec_null": null_spec,
    "spec_full_mean_cv": np.mean(outer_specificity),
    "spec_full_sd_cv": np.std(outer_specificity),
    "spec_diff": np.mean(outer_specificity) - null_spec,

    "acc_null": null_acc,
    "acc_full_mean_cv": np.mean(outer_accuracy),
    "acc_full_sd_cv": np.std(outer_accuracy),
    "acc_diff": np.mean(outer_accuracy) - null_acc
}])

print("\n Performance summary:")
print(performance_results)

# Optional save
performance_results.to_csv("/home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_nestedcv.txt", sep="\t", index=False)

### Test in PPMI and HBS

In [ ]:
# =========================
# Load and join data from PDBP
# =========================
clinical_path = "/home/jupyter/multiTRS/pheno/amp_pd_PPMI_PDBP_HBS_filtered_case_control_w.demographics.txt"
TRS_path = "/home/jupyter/multiTRS/scores/PDBP_TWAS_SMR_FDR_scores_resid.txt"

clinical = pd.read_csv(clinical_path, sep="\t")[["participant_id", "case_control_other_at_baseline", "sex", "age_at_baseline"]]

TRS = pd.read_csv(TRS_path, sep="\t")
TRS_cols = [c for c in TRS.columns if "Hip_fracture_EUR_2022_FDR_SMR_single_SNP" not in c]
TRS = TRS[TRS_cols]

combined_data = pd.merge(clinical, TRS, on="participant_id", how="inner")

print(f"Rows in combined PDBP data: {len(combined_data)}")

X = combined_data.drop(columns=["participant_id", "case_control_other_at_baseline"])
y = combined_data["case_control_other_at_baseline"]

print(f"Number of predictors: {X.shape[1]}")

# =========================
# Load saved hyperparameters and aggregate
# =========================
best_params_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_nestedcv_hyperparameters_all_outer.txt", sep="\t")

performance_df = pd.read_csv("/home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_nestedcv.txt", sep="\t")
print(f"\nLoaded mean nested CV AUC: {performance_df['AUC_full_mean_cv'].values[0]:.4f} SD: {performance_df['AUC_full_sd_cv'].values[0]:.4f}")

final_params = {
    'n_estimators':    int(mode(best_params_df['n_estimators'],    keepdims=True).mode[0]),
    'max_depth':       int(mode(best_params_df['max_depth'],       keepdims=True).mode[0]),
    'min_child_weight':int(mode(best_params_df['min_child_weight'],keepdims=True).mode[0]),
    'subsample':       best_params_df['subsample'].median(),
    'colsample_bytree':best_params_df['colsample_bytree'].median(),
    'learning_rate':   best_params_df['learning_rate'].median(),
    'gamma':           best_params_df['gamma'].median()
}

print("\nAggregated final hyperparameters:", final_params)

# =========================
# Fit final XGBoost model on full PDBP
# =========================
final_model = XGBClassifier(
    eval_metric='auc',
    random_state=1,
    **final_params
)

final_model.fit(X, y)

print("\nFinal XGBoost model fitted on full PDBP dataset.")

y_pred_prob = final_model.predict_proba(X)[:, 1]
final_auc = roc_auc_score(y, y_pred_prob)
print(f"Final XGBoost model AUC on full PDBP dataset: {final_auc:.4f}")

# =========================
# Extract and save SHAP values
# =========================
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)
base_value = explainer.expected_value

print("\nSHAP values extracted from final XGBoost model.")
print(f"SHAP values shape: {shap_values.shape}")
print(f"Base value (expected model output): {base_value:.6f}")

shap_df = pd.DataFrame(shap_values, columns=X.columns)
shap_df.to_csv("/home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_PDBP_SHAP_values.txt",
               sep="\t", index=False)
print("SHAP values saved to: /home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_PDBP_SHAP_values.txt")

print("\nGenerating SHAP bar plot...")
plt.figure()
shap.summary_plot(shap_values, X, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_all_scores_PDBP_SHAP_bar_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("SHAP bar plot saved to: /home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_PDBP_SHAP_bar_plot.png")

print("\nGenerating SHAP beeswarm plot...")
plt.figure()
shap.summary_plot(shap_values, X, show=False)
plt.tight_layout()
plt.savefig("/home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_PDBP_SHAP_beeswarm_plot.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Beeswarm plot saved to: /home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_PDBP_SHAP_beeswarm_plot.png")

# =========================
# Fit baseline logistic regression model on PDBP (sex + age)
# =========================
X_baseline = combined_data[["sex", "age_at_baseline"]].copy()

scaler = StandardScaler()
cols_to_scale_baseline = ["age_at_baseline"]
X_baseline_scaled = X_baseline.copy()
X_baseline_scaled[cols_to_scale_baseline] = scaler.fit_transform(X_baseline[cols_to_scale_baseline])

age_mean = scaler.mean_[0]
age_sd   = scaler.scale_[0]

baseline_model = LogisticRegression(
    random_state=1,
    max_iter=2000,
    penalty=None,
    solver='newton-cholesky',
    fit_intercept=True
)
baseline_model.fit(X_baseline_scaled, y)

print("\nBaseline logistic regression model fitted on full PDBP dataset.")
print(f"Age scaling parameters - Mean: {age_mean:.4f}, SD: {age_sd:.4f}")
print(f"Baseline model intercept: {baseline_model.intercept_[0]:.6f}")
print(f"Baseline model coefficients (sex, age): {baseline_model.coef_[0]}")

y_baseline_pred_prob = baseline_model.predict_proba(X_baseline_scaled)[:, 1]
baseline_auc = roc_auc_score(y, y_baseline_pred_prob)
print(f"Baseline (sex + age) model AUC on full PDBP dataset: {baseline_auc:.4f}")

# =========================
# Closest.topleft function (pROC equivalent)
# =========================
def closest_topleft_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
    idx = np.argmin(distances)
    return thresholds[idx], tpr[idx], 1 - fpr[idx]

def get_accuracy_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return accuracy_score(y_true, y_pred)

# =========================
# External validation in PPMI and HBS
# =========================
external_datasets = {
    "PPMI": {
        "TRS_path": "/home/jupyter/multiTRS/scores/PPMI_TWAS_SMR_FDR_scores_resid.txt"
    },
    "HBS": {
        "TRS_path": "/home/jupyter/multiTRS/scores/HBS_TWAS_SMR_FDR_scores_resid.txt"
    }
}

print("\n" + "="*70)
print("EXTERNAL VALIDATION RESULTS WITH DELONG'S TEST")
print("="*70)

full_results_summary = []

for dataset_name, paths in external_datasets.items():
    print(f"\n--- {dataset_name} Dataset ---")

    # Load TRS
    ext_TRS = pd.read_csv(paths["TRS_path"], sep="\t")
    TRS_cols = [c for c in ext_TRS.columns if "Hip_fracture_EUR_2022_FDR_SMR_single_SNP" not in c]
    ext_TRS = ext_TRS[TRS_cols]

    # Merge
    ext_combined = pd.merge(clinical, ext_TRS, on="participant_id", how="inner")
    print(f"Rows in combined data for {dataset_name}: {len(ext_combined)}")

    # Predictions - full model
    X_ext = ext_combined[X.columns]
    y_ext = ext_combined["case_control_other_at_baseline"]
    y_ext_pred_prob = final_model.predict_proba(X_ext)[:, 1]

    # Predictions - baseline model (scaled with PDBP parameters)
    X_ext_baseline = ext_combined[["sex", "age_at_baseline"]].copy()
    X_ext_baseline_scaled = X_ext_baseline.copy()
    X_ext_baseline_scaled["age_at_baseline"] = (X_ext_baseline["age_at_baseline"] - age_mean) / age_sd
    y_ext_baseline_pred_prob = baseline_model.predict_proba(X_ext_baseline_scaled)[:, 1]

    # DeLong test
    z_stat, p_value, ci_full, ci_null, auc_full, auc_null, info = Delong_test(
        y_ext, y_ext_pred_prob, y_ext_baseline_pred_prob,
        return_ci=True, return_auc=True, verbose=0
    )

    print(f"XGBoost model AUC:                   {auc_full:.4f} (95% CI: {ci_full[0]:.4f}-{ci_full[1]:.4f})")
    print(f"Baseline model AUC:                  {auc_null:.4f} (95% CI: {ci_null[0]:.4f}-{ci_null[1]:.4f})")
    print(f"AUC Difference (XGBoost - Baseline): {auc_full - auc_null:.4f}")
    print(f"Variance of AUC difference:          {info['var_diff']:.6f}")
    print(f"DeLong's test Z-statistic:           {z_stat:.4f}")
    print(f"DeLong's test p-value (2-tailed):    {p_value:.4e}")
    print(f"Result: {'Significantly different (p < 0.05)' if p_value < 0.05 else 'Not significantly different (p >= 0.05)'}")

    # Optimal threshold metrics
    thresh_full, sens_full, spec_full = closest_topleft_threshold(y_ext, y_ext_pred_prob)
    thresh_null, sens_null, spec_null = closest_topleft_threshold(y_ext, y_ext_baseline_pred_prob)

    acc_full = get_accuracy_at_threshold(y_ext, y_ext_pred_prob,          thresh_full)
    acc_null = get_accuracy_at_threshold(y_ext, y_ext_baseline_pred_prob, thresh_null)

    full_results_summary.append({
        'cohort':         dataset_name,
        'AUC_null':       auc_null,
        'AUC_lower_null': ci_null[0],
        'AUC_upper_null': ci_null[1],
        'AUC_full':       auc_full,
        'AUC_lower_full': ci_full[0],
        'AUC_upper_full': ci_full[1],
        'AUC_diff':       auc_full - auc_null,
        'sens_null':      sens_null,
        'sens_full':      sens_full,
        'sens_diff':      sens_full - sens_null,
        'spec_null':      spec_null,
        'spec_full':      spec_full,
        'spec_diff':      spec_full - spec_null,
        'acc_null':       acc_null,
        'acc_full':       acc_full,
        'acc_diff':       acc_full - acc_null,
        'delong_z':       z_stat,
        'delong_p':       p_value
    })

# =========================
# Save full results table
# =========================
print("\n" + "="*70)
print("FULL RESULTS TABLE")
print("="*70)
full_results_df = pd.DataFrame(full_results_summary)
print(full_results_df.to_string(index=False))

full_results_outfile = "/home/jupyter/multiTRS/results/multi_TRS_all_scores_XGBoost_external_validation_full_results.txt"
full_results_df.to_csv(full_results_outfile, sep="\t", index=False)
print(f"\nFull results saved to: {full_results_outfile}")

## Transfer results

In [ ]:
!ls /home/jupyter/multiTRS/results/
shell_do(f'gsutil -u {BILLING_PROJECT_ID} -m cp -r /home/jupyter/multiTRS/results/*XGBoost* {WORKSPACE_BUCKET}')